In [ ]:
import sys
import xarray as xr
import numpy as np
import pandas as pd
import math
import glob
import yaml
import geopandas as gpd
import cartopy
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colorbar import Colorbar # different way to handle colorbar
import matplotlib.ticker as mticker
import cmocean.cm as cmo

# cartopy
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
import cartopy.feature as cfeature
import dask

# import personal modules
# Path to modules
sys.path.append('../modules')
# Import my modules
import global_vars
from utils import roundPartial, select_months_ds
from plotter import draw_basemap, plot_terrain, plot_arscale_cbar
from colorline import colorline
from trajectory_post_funcs import calculate_heatmaps_from_trajectories
from load_trajectories import load_trajectories_based_on_region
from load_shapefiles import load_region_shp, load_HUC8
import customcmaps as ccmap

dask.config.set(**{'array.slicing.split_large_chunks': True})

In [ ]:
path_to_data = global_vars.path_to_data
path_to_out  = '../out/'       # output files (numerical results, intermediate datafiles) -- read & write
path_to_figs = '../figs/'      # figures

In [ ]:

HUC8_ID_lst = [14050001, ## upper yampa
               14010001, ## roaring fork
               14020002, ## upper gunnison
               14080101, ## upper san juan
               # 14050005, ## upper white
               # 14050002, ## lower yampa
               # 14080104, ## animas (San Juans),
               # 14030002, ## upper dolores
               # 11020002, ## arkansas - Pueblo Reservoir
               # 10190005 ## St. Vrain (Boulder)
               # 14030005, ## 'Upper Colorado-Kane Springs'
               # 10190002, ## 'Upper South Platte'
               # 10190018, ## 'Lower South Platte'
               # 10190012, ## 'Middle South Platte-Sterling'
               # 11020001 ## Arkansas Headwaters
               # 11020009 ## Upper Arkansas-John Martin Reservoir
              ]

In [ ]:
start_mon = 11
end_mon = 4

## load trajectories
ds = load_trajectories_based_on_region()
ds = select_months_ds(ds, start_mon, end_mon, 'start_date')
ds


In [ ]:
polys = load_HUC8()
# regions = load_region_shp(polys)

In [ ]:
# Set up projection
datacrs = ccrs.PlateCarree()  ## the projection the data is in
mapcrs = ccrs.PlateCarree() ## the projection you want your map displayed in

ext = [-140., -90., 20, 50]

# Set tick/grid locations
tx = 10
ty = 5
dx = np.arange(ext[0],ext[1]+tx,tx)
dy = np.arange(ext[2],ext[3]+ty,ty)

titlestring = [['(a)', '(b)', '(c)', '(d)'],
               ['(e)', '(f)', '(g)', '(h)']]

In [ ]:
nrows = 6
ncols = 2

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.1)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(7.75, 10.))
fig.dpi = 600
fname = path_to_figs + 'tARget_trajectory-IVT_heatmaps_noAR'
fmt = 'png'

#####################
### PLOT HEATMAPS ###
#####################
basin_lst = ['Upper Yampa', 'Roaring Fork', 'Upper Gunnison', 'Upper San Juan']
# Add color bar axis
cbax = plt.subplot(gs[-1,0]) # colorbar axis

col_idx = [0, 0, 0, 0]
row_idx = [0, 1, 2, 3]
blon_lst = [False, False, False, True]
for i, HUC8 in enumerate(HUC8_ID_lst):
    ax = fig.add_subplot(gs[row_idx[i],col_idx[i]], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy,left_lats=True, 
                      right_lats=False, bottom_lons=blon_lst[i])
    
    ax.set_extent(ext, datacrs)
    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
    ax.text(-0.16, 0.5, basin_lst[i], va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', fontsize=13,
                transform=ax.transAxes)
    
    tmp = ds.where(ds.HUC8==str(HUC8), drop=True).squeeze()
    AR = tmp.where(tmp.tARget.isnull(), drop=True)
    ## now calculate heatmaps from remaining trajectories
    cell = calculate_heatmaps_from_trajectories(AR, ARDT='tARgetv4', normalize=False, AR=False)

    ## create segmented cmap
    cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 55, 5))
    ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
    cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                  legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})


    ## add in four focus subbasins
    plot_poly = polys[(polys.HUC8 == str(HUC8_ID_lst[i]))]
    plot_poly.crs = 'epsg:3857'
    print(plot_poly.crs)
    plot_poly.plot(ax=ax, edgecolor='white', color='None', zorder=99, lw=0.5)

    ax.text(0.03, 0.96, titlestring[0][i], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)

#######################
### PLOT IVT VALUES ###
#######################
col_idx = [1, 1, 1, 1]
row_idx = [0, 1, 2, 3]
blon_lst = [False, False, False, True]
for i, HUC8 in enumerate(HUC8_ID_lst):
    ax = fig.add_subplot(gs[row_idx[i], col_idx[i]], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=False, 
                      right_lats=False, bottom_lons=blon_lst[i])
    ax.set_extent(ext, datacrs)
    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
    
    tmp = ds.where(ds.HUC8==str(HUC8), drop=True)
    AR = tmp.where(tmp.tARget.isnull(), drop=True)
    nevents = len(AR.start_date)
    ## Add different points
    for j in range(nevents):
        data = AR.isel(start_date=j)
        y_lst = data.lat.values
        x_lst = data.lon.values
        z_lst = data.IVT.values
        # ax.plot(x_lst, y_lst, c='gray', transform=datacrs, alpha=0.2)
        cmap, norm, bnds = ccmap.cmap('ivt')
        cf = ax.scatter(x_lst, y_lst, c=z_lst, cmap=cmap, norm=norm, marker='.', transform=datacrs, alpha=0.8, s=6)

    ## add in four focus subbasins
    plot_poly = polys[(polys.HUC8 == str(HUC8_ID_lst[i]))]
    plot_poly.crs = 'epsg:3857'
    print(plot_poly.crs)
    plot_poly.plot(ax=ax, edgecolor='black', color='None', zorder=99, lw=0.5)

    ax.text(0.03, 0.96, titlestring[1][i], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)


# Add color bar
cbarticks = [150, 250, 400, 600, 800, 1200, 1600]
cbax = plt.subplot(gs[-1,-1]) # colorbar axis
cb = Colorbar(ax = cbax, mappable = cf, orientation = 'horizontal', ticklocation = 'bottom', ticks=cbarticks)
cb.set_label('IVT (kg m$^{-1}$ s$^{-1}$)', fontsize=10)
cb.ax.tick_params(labelsize=10)

fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()

In [ ]:
nrows = 6
ncols = 2

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.01, hspace=0.1)
## use gs[rows index, columns index] to access grids

fig = plt.figure(figsize=(7.75, 10.))
fig.dpi = 600
fname = path_to_figs + 'tARget_trajectory_heatmaps_AR'
fmt = 'png'

#####################
### PLOT HEATMAPS ###
#####################
basin_lst = ['Upper Yampa', 'Roaring Fork', 'Upper Gunnison', 'Upper San Juan']
# Add color bar axis
cbax = plt.subplot(gs[-1,0]) # colorbar axis

col_idx = [0, 0, 0, 0]
row_idx = [0, 1, 2, 3]
blon_lst = [False, False, False, True]
for i, HUC8 in enumerate(HUC8_ID_lst):
    ax = fig.add_subplot(gs[row_idx[i],col_idx[i]], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy,left_lats=True, 
                      right_lats=False, bottom_lons=blon_lst[i])
    
    ax.set_extent(ext, datacrs)
    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
    ax.text(-0.16, 0.5, basin_lst[i], va='bottom', ha='center',
                rotation='vertical', rotation_mode='anchor', fontsize=13,
                transform=ax.transAxes)
    
    tmp = ds.where(ds.HUC8==str(HUC8), drop=True).squeeze()
    AR = tmp.where(tmp.tARget > 0, drop=True)
    ## now calculate heatmaps from remaining trajectories
    cell = calculate_heatmaps_from_trajectories(AR, ARDT='tARgetv4', normalize=False, AR=False)

    ## create segmented cmap
    cmap, norm, bnds = ccmap.cmap_segmented(cmo.deep, np.arange(0, 35, 5))
    ## plotting based off of https://geopandas.org/en/stable/docs/user_guide/mapping.html
    cf = cell.plot(ax=ax, column='n_traj', cmap=cmap, vmin=bnds[0], vmax=bnds[-1], norm=norm, edgecolor=None, legend=True, cax=cbax,
                  legend_kwds={"label": "Trajectory Frequency (count)", "orientation": "horizontal"})
    
    ## add in four focus subbasins
    plot_poly = polys[(polys.HUC8 == str(HUC8_ID_lst[i]))]
    plot_poly.crs = 'epsg:3857'
    print(plot_poly.crs)
    plot_poly.plot(ax=ax, edgecolor='white', color='None', zorder=99, lw=0.5)

    ax.text(0.03, 0.96, titlestring[0][i], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)

#######################
### PLOT IVT VALUES ###
#######################
colors = ['#F5F0E6', '#0ac1ff', '#04ff03', '#ffff03', '#ffa602', '#ff0100']
col_idx = [1, 1, 1, 1]
row_idx = [0, 1, 2, 3]
blon_lst = [False, False, False, True]
for i, HUC8 in enumerate(HUC8_ID_lst):
    ax = fig.add_subplot(gs[row_idx[i], col_idx[i]], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext, xticks=dx, yticks=dy, left_lats=False, 
                      right_lats=False, bottom_lons=blon_lst[i])
    ax.set_extent(ext, datacrs)
    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8)
    
    tmp = ds.where(ds.HUC8==str(HUC8), drop=True)
    ## Loop through AR scale values
    for k in range(1, 7):
        try:
            AR = tmp.where(tmp.ar_scale == k, drop=True)
            nevents = len(AR.start_date)
            ## LOOP THROUGH TRAJECTORIES
            for m in range(nevents):
                data = AR.isel(start_date=m)
                y_lst = data.lat.values
                x_lst = data.lon.values
                ax.plot(x_lst, y_lst, c=colors[k-1], transform=datacrs, alpha=0.2)
                cf = ax.scatter(x_lst, y_lst, c=colors[k-1], marker='.', transform=datacrs, alpha=0.7, s=6)
        except IndexError:
            pass
            
    ## add in four focus subbasins
    plot_poly = polys[(polys.HUC8 == str(HUC8_ID_lst[i]))]
    plot_poly.crs = 'epsg:3857'
    print(plot_poly.crs)
    plot_poly.plot(ax=ax, edgecolor='black', color='None', zorder=99, lw=0.5)

    ax.text(0.03, 0.96, titlestring[1][i], ha='left', va='top', transform=ax.transAxes, fontsize=11., backgroundcolor='white', zorder=101)


## Add color bar
cbax = plt.subplot(gs[-1,-1]) # colorbar axis
plot_arscale_cbar(cbax, orientation='horizontal')

fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi, transparent=True)
plt.show()
fig.clf()